<a href="https://colab.research.google.com/github/rithikanarendranwork-pixel/Digital-Building-footprint/blob/main/vector_to_raster_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install gdal


In [ ]:
from pyproj import Transformer

# Define the source (Projected CRS) and target (Geographic CRS)
source_crs = "EPSG:32643"  # Example: UTM Zone 43N
target_crs = "EPSG:4326"   # WGS 84 (Lat/Lon)

# Initialize Transformer
transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)

# Given UTM coordinates
utm_x, utm_y = 773694.7, 1433151.1

# Convert to Latitude, Longitude
lon, lat = transformer.transform(utm_x, utm_y)

print(f"Latitude: {lat}, Longitude: {lon}")

In [ ]:
from osgeo import gdal, ogr

#paths to the vector and raster files
vector_file = '/content/drive/MyDrive/Bang_Part/Bengaluru_Part.shp'  # Ensure this file is in the correct path
output_raster = '/content/drive/MyDrive/Bang_Part/Bengaluru_Part2.tif'  # Output raster file
pixel_size = 0.6  # (check info in qgis )
no_data_value = 0  # Background value
vector_ds = ogr.Open(vector_file)
layer = vector_ds.GetLayer()

#x_min, x_max, y_min, y_max = layer.GetExtent()
x_min, x_max, y_min, y_max = 777729.3930981999728829,781027.6866934000281617,1427441.4396319999359548 ,1435310.0054311999119818 #manually assigned them pased on *.tif file
print(x_min,x_max, y_min, y_max) #debugging
x_res = int((x_max - x_min) / pixel_size) #width
y_res = int((y_max - y_min) / pixel_size) #height
print(x_res,y_res)
driver = gdal.GetDriverByName("GTiff")
raster_ds = driver.Create(output_raster, x_res, y_res, 1, gdal.GDT_Byte)


raster_ds.SetGeoTransform((x_min, pixel_size, 0, y_max, 0, -pixel_size)) #position
print(layer)  #debugging purpose

raster_ds.SetProjection(layer.GetSpatialRef().ExportToWkt()) #Set projection (same as vector file)

#  Rasterize only the **XY** part of MultiPolygonZ (ignoring Z)
gdal.RasterizeLayer(raster_ds, [1], layer, burn_values=[1])  # 1 White pixels
raster_ds = None
vector_ds = None #close the raster and vector ds

print(f" Raster file saved as: {output_raster}")
